# Fine-Tune Qwen2.5-1.5B for Football Tactical Analysis

Uses **LlamaFactory** (same approach as `ocr_finetune_vlm.ipynb`).

- **Method**: LoRA via LlamaFactory CLI + QLoRA 4-bit
- **Dataset**: `data/finetune/football_tactical.jsonl` (ShareGPT chat format)
- **GPU**: GTX 1660 Ti 6GB or Colab T4

---
### To switch to a different model later
Only change `MODEL_NAME_OR_PATH` in **Section 3** and `template` in **Section 4**. Everything else is reusable.

## 1. Installation

In [ ]:
# Same versions as ocr_finetune_vlm.ipynb reference
!pip install transformers==4.57.6
!pip install optimum==1.26.0
!pip install datasets==4.4.0

!pip install torch==2.8.0
!pip install torchvision==0.23
!pip install torchaudio==2.8.0

# Clone LlamaFactory (pinned to same commit as reference)
!git clone --depth 1 https://github.com/hiyouga/LlamaFactory.git
!cd LlamaFactory && git checkout 762b480131908d37736ad9aa3f12e87f8f7e6313

!cd LlamaFactory && pip install -e .
!cd LlamaFactory && pip install -r requirements/metrics.txt

## 2. Prepare Dataset — Convert JSONL → ShareGPT JSON

LlamaFactory expects a **JSON array** in ShareGPT format:
```json
[{"conversations": [{"from": "human", "value": "..."}, {"from": "gpt", "value": "..."}]}, ...]
```
Our `football_tactical.jsonl` already has `messages` with `role/content` — we just convert.

In [ ]:
import json, os, random

# ── Paths ────────────────────────────────────────────────────────────────────
JSONL_PATH  = "../data/finetune/football_tactical.jsonl"   # source JSONL
TRAIN_JSON  = "football_train.json"                        # output train
VAL_JSON    = "football_val.json"                          # output val
VAL_SPLIT   = 0.10                                         # 10% validation
SEED        = 42

# ── Load JSONL ───────────────────────────────────────────────────────────────
records = []
with open(JSONL_PATH, encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        rec = json.loads(line)
        messages = rec.get("messages", [])

        # Convert role/content → from/value  (ShareGPT format)
        role_map = {"user": "human", "assistant": "gpt", "system": "system"}
        conversations = [
            {"from": role_map.get(m["role"], m["role"]), "value": m["content"]}
            for m in messages
        ]
        records.append({"conversations": conversations})

print(f"Total records: {len(records)}")

# ── Train / Val split ────────────────────────────────────────────────────────
random.Random(SEED).shuffle(records)
val_n       = max(1, int(len(records) * VAL_SPLIT))
val_ds      = records[:val_n]
train_ds    = records[val_n:]

with open(TRAIN_JSON, "w", encoding="utf-8") as f:
    json.dump(train_ds, f, ensure_ascii=False)
with open(VAL_JSON, "w", encoding="utf-8") as f:
    json.dump(val_ds,   f, ensure_ascii=False)

print(f"Train: {len(train_ds)} | Val: {len(val_ds)}")
print(f"\nSample conversation:")
print(json.dumps(train_ds[0]["conversations"][0], indent=2))

## 3. Register Dataset in LlamaFactory

Add entries to `LlamaFactory/data/dataset_info.json` so the CLI can find our files.

In [ ]:
import json, os

TRAIN_ABS = os.path.abspath(TRAIN_JSON)
VAL_ABS   = os.path.abspath(VAL_JSON)

info_path = "LlamaFactory/data/dataset_info.json"
with open(info_path, encoding="utf-8") as f:
    dataset_info = json.load(f)

# ── Add our two splits ────────────────────────────────────────────────────────
dataset_info["football_train"] = {
    "file_name": TRAIN_ABS,
    "formatting": "sharegpt",
    "columns": {"messages": "conversations"}
}
dataset_info["football_val"] = {
    "file_name": VAL_ABS,
    "formatting": "sharegpt",
    "columns": {"messages": "conversations"}
}

with open(info_path, "w", encoding="utf-8") as f:
    json.dump(dataset_info, f, indent=2, ensure_ascii=False)

print(f"✓ Registered 'football_train' and 'football_val'")
print(f"  Train file: {TRAIN_ABS}")
print(f"  Val   file: {VAL_ABS}")

## 4. Write Training YAML Config

The YAML controls the model, LoRA settings, dataset, and training hyper-parameters.

> **To add a new model**: change `model_name_or_path` and `template`. That is all.

In [2]:
import os

# ── ⚙️  CHANGE ONLY THIS BLOCK TO SWITCH MODELS ──────────────────────────────
MODEL_NAME_OR_PATH = "Qwen/Qwen2.5-1.5B-Instruct"   # HuggingFace model id
TEMPLATE           = "qwen"                           # LlamaFactory template name
OUTPUT_DIR         = "../models/finetuned_qwen25"     # where checkpoints go
# ─────────────────────────────────────────────────────────────────────────────

os.makedirs(OUTPUT_DIR, exist_ok=True)
YAML_PATH = "LlamaFactory/examples/train_lora/football_qwen25.yaml"
os.makedirs(os.path.dirname(YAML_PATH), exist_ok=True)

yaml_content = f"""
### model
model_name_or_path: {MODEL_NAME_OR_PATH}
use_fast_tokenizer: false
cache_dir: ../models/cache
trust_remote_code: true

### method
stage: sft
do_train: true
finetuning_type: lora
lora_rank: 16
lora_alpha: 32
lora_dropout: 0.05
lora_target: all
use_rslora: true

### dataset
dataset: football_train
eval_dataset: football_val
template: {TEMPLATE}
cutoff_len: 1024
overwrite_cache: true
preprocessing_num_workers: 1

### output
# resume_from_checkpoint: {os.path.abspath(OUTPUT_DIR)}/checkpoint-100
output_dir: {os.path.abspath(OUTPUT_DIR)}
logging_steps: 10
save_steps: 50
save_total_limit: 3
plot_loss: true
overwrite_output_dir: true

### train
per_device_train_batch_size: 1
gradient_accumulation_steps: 8
learning_rate: 2.0e-4
num_train_epochs: 3.0
lr_scheduler_type: cosine
warmup_ratio: 0.05
fp16: true
ddp_timeout: 180000000

### eval
per_device_eval_batch_size: 1
eval_strategy: steps
eval_steps: 50

report_to: wandb
run_name: yt-football-finetune-llamafactory2
""".strip()

with open(YAML_PATH, "w") as f:
    f.write(yaml_content)


print(f"✓ YAML written to: {YAML_PATH}")
print()
print(yaml_content)

✓ YAML written to: LlamaFactory/examples/train_lora/football_qwen25.yaml

### model
model_name_or_path: Qwen/Qwen2.5-1.5B-Instruct
use_fast_tokenizer: false
cache_dir: ../models/cache
trust_remote_code: true

### method
stage: sft
do_train: true
finetuning_type: lora
lora_rank: 16
lora_alpha: 32
lora_dropout: 0.05
lora_target: all
use_rslora: true

### dataset
dataset: football_train
eval_dataset: football_val
template: qwen
cutoff_len: 1024
overwrite_cache: true
preprocessing_num_workers: 1

### output
# resume_from_checkpoint: d:\SASUniversityEdition\Machine\MODEL\football\models\finetuned_qwen25/checkpoint-100
output_dir: d:\SASUniversityEdition\Machine\MODEL\football\models\finetuned_qwen25
logging_steps: 10
save_steps: 50
save_total_limit: 3
plot_loss: true
overwrite_output_dir: true

### train
per_device_train_batch_size: 1
gradient_accumulation_steps: 8
learning_rate: 2.0e-4
num_train_epochs: 3.0
lr_scheduler_type: cosine
warmup_ratio: 0.05
fp16: true
ddp_timeout: 180000000

###

## 5. Login to HuggingFace (Qwen2.5 is public — only needed for gated models)

In [ ]:
# Uncomment if running on Colab with userdata
# from google.colab import userdata
# hf_token = userdata.get('huggingface')
# !huggingface-cli login --token {hf_token}

# Or paste your token directly:
# !huggingface-cli login --token hf_YOUR_TOKEN_HERE

print("Qwen2.5-1.5B-Instruct is public — no token needed.")
print("Uncomment the lines above if you use a private/gated model.")

## 6. Train

In [3]:
import subprocess, os

yaml_abs = os.path.abspath(YAML_PATH)

os.environ["DISABLE_VERSION_CHECK"] = "1"
!cd LlamaFactory && llamafactory-cli train {yaml_abs}

^C


## 7. Inference with Fine-Tuned Model

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
import torch

LORA_PATH = os.path.abspath(OUTPUT_DIR)

# Load base model + LoRA adapter
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME_OR_PATH, trust_remote_code=True)
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME_OR_PATH,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True,
)
model = PeftModel.from_pretrained(base_model, LORA_PATH)
model.eval()

def generate_analysis(prompt: str, max_new_tokens: int = 512) -> str:
    """
    Generate tactical analysis from fine-tuned Qwen2.5.
    To swap models: just change MODEL_NAME_OR_PATH and LORA_PATH above.
    """
    messages = [{"role": "user", "content": prompt}]
    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.7,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
        )
    return tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)


# ── Test ──────────────────────────────────────────────────────────────────────
test_prompt = """Analyze the upcoming match: Manchester City (Home) vs Arsenal (Away).

Manchester City recent form (last 5 matches):
  - Points per match (last 5): 2.20
  - Goals scored per match (last 5): 3.20
  - Goals conceded per match (last 5): 0.80
  - Shots on target per match (last 5): 7.40

Arsenal recent form (last 5 matches):
  - Points per match (last 5): 1.80
  - Goals scored per match (last 5): 2.10
  - Goals conceded per match (last 5): 1.20
  - Shots on target per match (last 5): 5.80

Head-to-Head: Total meetings: 10, Home wins: 4, Away wins: 2, Draws: 4

Prediction: Home Win (Home 58.3% | Draw 24.1% | Away 17.6%)"""

print("=" * 70)
result = generate_analysis(test_prompt)
print(result)

## 8. Integration with `llm_providers.py`

Paste this class into `models/llm_providers.py` and add it to the provider factory:

```python
class FinetunedLlamaFactoryProvider:
    """Provider for any model fine-tuned via LlamaFactory + LoRA."""
    def __init__(self, base_model_id: str, lora_path: str):
        from transformers import AutoModelForCausalLM, AutoTokenizer
        from peft import PeftModel
        import torch
        self.tokenizer = AutoTokenizer.from_pretrained(base_model_id, trust_remote_code=True)
        base = AutoModelForCausalLM.from_pretrained(
            base_model_id, torch_dtype=torch.float16,
            device_map="auto", trust_remote_code=True
        )
        self.model = PeftModel.from_pretrained(base, lora_path).eval()

    def generate_explanation(self, match_context: dict, gnn_explanation: dict) -> str:
        prompt = self._build_prompt(match_context, gnn_explanation)
        messages = [{"role": "user", "content": prompt}]
        text = self.tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )
        inputs = self.tokenizer(text, return_tensors="pt").to(self.model.device)
        with torch.no_grad():
            out = self.model.generate(**inputs, max_new_tokens=512,
                                      temperature=0.7, do_sample=True)
        return self.tokenizer.decode(
            out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True
        )
```

Register it:
```python
"qwen-finetuned": lambda: FinetunedLlamaFactoryProvider(
    base_model_id = "Qwen/Qwen2.5-1.5B-Instruct",
    lora_path     = "models/finetuned_qwen25",
)
```